## RecDistillery: A Framework for Teacher-Student Knowledge Distillation in Recommender Systems

This is a compact Google Colab notebook to test the framework and to reproduce our four case study experiments:

1. Elliot LGCN: Teacher, Student, DE
2. RecBole SGL: Teacher, Student, RRD
3. LensKit BPRMF: Teacher, Student, FTD
4. External ItemKNN Teacher, Elliot NMF Student, UnKD

In Colab, select `Runtime > Change runtime type > GPU` before running the notebook, to enable the fastest computational speed.

Note that the experiments might take some time, as we explore some hyperparameter ranges via Bayesian optimization with 
30 trials, trained for up to 300 epoch, for each run. To run faster experiments, override these default values ​​in the 
configuration files in `config/experiments`, under the block `optimization`.

### 1. Clone Repository

In [ ]:
!git clone https://github.com/Mari-eng02/RecDistillery.git
%cd RecDistillery

### 2. Install Dependencies

This cell uses the project's CUDA requirements file, suitable for Colab GPU runtimes.

In [ ]:
!pip install -r setup/requirements_cuda.txt

### 3. Prepare CiteULike

All four experiments below use CiteULike, so this notebook prepares only this dataset.

In [ ]:
%cd /content/RecDistillery
%env PYTHONPATH=/content/RecDistillery
!bash setup/setup_directories.sh
!python scripts/data_preparation/citeulike.py

### 4. Small Helpers

`latest()` is only used to retrieve the most recent artifact generated after each training/import step.

In [ ]:
from pathlib import Path
import json
import pandas as pd

TOP_K = 20

def latest(pattern):
    matches = sorted(Path('.').glob(pattern), key=lambda p: p.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"No artifact found for pattern: {pattern}")
    path = str(matches[-1])
    print(path)
    return path

### 5. Experiment 1 - Elliot LGCN + DE

Reproduce the CiteULike experiment with an Elliot LightGCN teacher, an Elliot LightGCN plain student, and DE distillation.

This reproduces the native Elliot LGCN teacher run. The distillation config below is pinned to the artifact used in the original experiment; update the `teacher.path` in the distillation config file to distill from the freshly generated teacher.

In [ ]:
!python scripts/teacher_training/teacher_training.py --config config/experiments/teacher/elliot_lgcn_citeulike_001.yaml
ELLIOT_LGCN_TEACHER = latest('results/teacher/**/artifacts/*elliot*LGCN*citeulike*best.teacher')
!python scripts/recdistill/evaluate_teacher.py --teacher-path {ELLIOT_LGCN_TEACHER} --top-k {TOP_K}

In [ ]:
!python scripts/student_training/student_training.py --config config/experiments/student/elliot_lgcn_citeulike_001.yaml
ELLIOT_LGCN_STUDENT = latest('results/student/**/artifacts/*elliot*LGCN*citeulike*best.student')
!python scripts/recdistill/evaluate_students.py --student-path {ELLIOT_LGCN_STUDENT} --top-k {TOP_K}

In [ ]:
!python scripts/recdistill/train_student_from_config.py --config config/experiments/recdistill/de_citeulike_001.yaml --teacher-path {ELLIOT_LGCN_TEACHER}
DE_STUDENT = latest('results/recdistill/**/artifacts/*de*citeulike*best.distilled_student')
!python scripts/recdistill/evaluate_students.py --student-path {DE_STUDENT} --top-k {TOP_K}

### 6. Experiment 2 - LensKit BPRMF + FTD

Reproduce the CiteULike experiment with a LensKit BPRMF teacher, a LensKit BPRMF plain student, and FTD distillation.

The FTD config is pinned to the original teacher artifact. After generating the new teacher, update `teacher.path` in the distillation config file.

In [ ]:
!python scripts/teacher_training/teacher_training.py --config config/experiments/teacher/lenskit_bprmf_citeulike_002.yaml
LENSKIT_BPRMF_TEACHER = latest('results/teacher/**/artifacts/*lenskit*BPRMF*citeulike*best.teacher')
!python scripts/recdistill/evaluate_teacher.py --teacher-path {LENSKIT_BPRMF_TEACHER} --top-k {TOP_K}

In [ ]:
!python scripts/student_training/student_training.py --config config/experiments/student/lenskit_bprmf_citeulike_002.yaml
LENSKIT_BPRMF_STUDENT = latest('results/student/**/artifacts/*lenskit*BPRMF*citeulike*best.student')
!python scripts/recdistill/evaluate_students.py --student-path {LENSKIT_BPRMF_STUDENT} --top-k {TOP_K}

In [ ]:
!python scripts/recdistill/train_student_from_config.py --config config/experiments/recdistill/ftd_citeulike_002.yaml --teacher-path {LENSKIT_BPRMF_TEACHER}
FTD_STUDENT = latest('results/recdistill/**/artifacts/*ftd*citeulike*best.distilled_student')
!python scripts/recdistill/evaluate_students.py --student-path {FTD_STUDENT} --top-k {TOP_K}

### 7. Experiment 3 - RecBole SGL + RRD

Reproduce the CiteULike experiment with a RecBole SGL teacher, a RecBole SGL plain student, and RRD distillation.

The RRD config is pinned to the original teacher artifact. After generating the new teacher, update `teacher.path` in RRD config file.

In [ ]:
!python scripts/teacher_training/teacher_training.py --config config/experiments/teacher/recbole_sgl_citeulike_003.yaml
RECBOLE_SGL_TEACHER = latest('results/teacher/**/artifacts/*recbole*SGL*citeulike*best.teacher')
!python scripts/recdistill/evaluate_teacher.py --teacher-path {RECBOLE_SGL_TEACHER} --top-k {TOP_K}

In [ ]:
!python scripts/student_training/student_training.py --config config/experiments/student/recbole_sgl_citeulike_003.yaml
RECBOLE_SGL_STUDENT = latest('results/student/**/artifacts/*recbole*SGL*citeulike*best.student')
!python scripts/recdistill/evaluate_students.py --student-path {RECBOLE_SGL_STUDENT} --top-k {TOP_K}

In [ ]:
!python scripts/recdistill/train_student_from_config.py --config config/experiments/recdistill/rrd_citeulike_003.yaml --teacher-path {RECBOLE_SGL_TEACHER}
RRD_STUDENT = latest('results/recdistill/**/artifacts/*rrd*citeulike*best.distilled_student')
!python scripts/recdistill/evaluate_students.py --student-path {RRD_STUDENT} --top-k {TOP_K}

### 8. Experiment 4 - External ItemKNN + Elliot NMF + UnKD

Reproduce the CiteULike experiment with an externally imported Elliot ItemKNN teacher, an Elliot NMF plain student, and UnKD distillation.

Before running this section, make sure the external export is available at: `external/Elliot_ItemKNN/teacher_itemknn.json`

The file must contain recommendations whose IDs match the original CiteULike splits, or explicit mapping metadata.

In [ ]:
EXTERNAL_ITEMKNN_JSON = 'external/Elliot_ItemKNN/teacher_itemknn.json'
Path(EXTERNAL_ITEMKNN_JSON).exists()

In [ ]:
!python scripts/recdistill/import_teacher.py \
  --input {EXTERNAL_ITEMKNN_JSON} \
  --format predictions_json \
  --framework external \
  --model-name ItemKNN \
  --dataset citeulike

ITEMKNN_TEACHER = latest('results/teacher/**/artifacts/*external*ItemKNN*citeulike*best.teacher')
!python scripts/recdistill/evaluate_teacher.py --teacher-path {ITEMKNN_TEACHER} --top-k {TOP_K}

In [ ]:
!python scripts/student_training/student_training.py --config config/experiments/student/elliot_nmf_citeulike_004.yaml
ELLIOT_NMF_STUDENT = latest('results/student/**/artifacts/*elliot*NMF*citeulike*best.student')
!python scripts/recdistill/evaluate_students.py --student-path {ELLIOT_NMF_STUDENT} --top-k {TOP_K}

In [ ]:
!python scripts/recdistill/train_student_from_config.py --config config/experiments/recdistill/unkd_citeulike_004.yaml --teacher-path {ITEMKNN_TEACHER}
UNKD_STUDENT = latest('results/recdistill/**/artifacts/*unkd*citeulike*best.distilled_student')
!python scripts/recdistill/evaluate_students.py --student-path {UNKD_STUDENT} --top-k {TOP_K}

### 9. Summary table

This cell collects the latest evaluation JSON files produced under `results/`.

In [ ]:
rows = []
for path in sorted(Path('results').glob('**/perf/*eval_top20.json')):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    rows.append({
        'file': str(path),
        'precision': data.get('precision'),
        'recall': data.get('recall'),
        'ndcg': data.get('ndcg'),
        'hr': data.get('hr'),
    })

pd.DataFrame(rows)